# Many body entagled systems


In [ ]:
using ITensors

spin = Index(2,"spin")

# W state - symmetric superposition of |100⟩, |010⟩, |001⟩
W = ITensor(ComplexF64, spin, spin', spin'')
W[spin=>2, spin'=>1, spin''=>1] = 1.0 / sqrt(3)
W[spin=>1, spin'=>2, spin''=>1] = 1.0 / sqrt(3)
W[spin=>1, spin'=>1, spin''=>2] = 1.0 / sqrt(3)

# GHZ state - superposition of |000⟩ and |111⟩
GHZ = ITensor(ComplexF64, spin, spin', spin'')
GHZ[spin=>1,spin'=>1, spin''=>1] = 1.0 / sqrt(2)
GHZ[spin=>2,spin'=>2, spin''=>2] = 1.0 / sqrt(2)

In [ ]:
# Let's examine the properties of these quantum states
println("W State:")
println("W state tensor: ")
display(W)
println("\nNormalization of W state: ", norm(W))

println("\n" * "="^50)
println("\nGHZ State:")
println("GHZ state tensor: ")
display(GHZ)
println("\nNormalization of GHZ state: ", norm(GHZ))

## Understanding the Quantum States

### W State
The **W state** is a 3-qubit entangled state representing an equal superposition of states where exactly one qubit is in the |1⟩ state and the others are in |0⟩:
$$|W\rangle = \frac{1}{\sqrt{3}}(|100\rangle + |010\rangle + |001\rangle)$$

Key properties:
- **Symmetric entanglement**: All qubits are equally entangled
- **Robust against particle loss**: If one qubit is lost, the remaining two are still entangled
- **Partial separability**: One qubit can be separated while maintaining entanglement between the other two

### GHZ State  
The **GHZ (Greenberger-Horne-Zeilinger) state** is a maximally entangled 3-qubit state:
$$|GHZ\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

Key properties:
- **Maximal entanglement**: All three qubits are maximally entangled
- **Fragile against particle loss**: If one qubit is lost, the remaining state becomes separable
- **Demonstrates quantum non-locality** more strongly than Bell states

In [ ]:
# Let's compute some entanglement measures and compare the states

# Function to get all non-zero elements from an ITensor
function show_nonzero_elements(T, name)
    println("Non-zero elements of $name:")
    tensor_inds = ITensors.inds(T)
    for i in 1:dim(tensor_inds[1]), j in 1:dim(tensor_inds[2]), k in 1:dim(tensor_inds[3])
        val = T[tensor_inds[1]=>i, tensor_inds[2]=>j, tensor_inds[3]=>k]
        if abs(val) > 1e-10
            println("  |$(i-1)$(j-1)$(k-1)⟩: $(val)")
        end
    end
end

show_nonzero_elements(W, "W state")
println()
show_nonzero_elements(GHZ, "GHZ state")

In [ ]:
# Creating Tensor Trains (Matrix Product States) with ITensors

# Define the number of sites and local dimension
N = 6  # number of sites

# Create a set of indices for the tensor train
sites = [Index(2, "S=1/2,Site,n=$i") for i in 1:N]

println("Created $N spin-1/2 sites:")
for (i, s) in enumerate(sites)
    println("Site $i: $s")
end

# Create bond indices for the tensor train
println("\n" * "="^50)
println("Creating bond indices for tensor train:")

χ = 4  # bond dimension
bonds = [Index(χ, "Link,l=$i") for i in 1:(N-1)]

println("Bond indices:")
for (i, b) in enumerate(bonds)
    println("Bond $i: $b")
end

In [ ]:
# Now let's create the actual tensor train tensors
println("Creating tensor train (MPS) tensors:")

# Create an array to store the MPS tensors
mps_tensors = ITensor[]

# First tensor: only has physical index and right bond
A1 = randomITensor(sites[1], bonds[1])
push!(mps_tensors, A1)
println("Tensor 1: $(inds(A1))")

# Middle tensors: have left bond, physical index, and right bond  
for i in 2:(N-1)
    Ai = randomITensor(bonds[i-1], sites[i], bonds[i])
    push!(mps_tensors, Ai)
    println("Tensor $i: $(inds(Ai))")
end

# Last tensor: only has left bond and physical index
AN = randomITensor(bonds[N-1], sites[N])
push!(mps_tensors, AN)
println("Tensor $N: $(inds(AN))")

println("\nTensor train created with $(length(mps_tensors)) tensors!")

# Calculate total parameters
total_params = 0
for (i, t) in enumerate(mps_tensors)
    params = prod([dim(idx) for idx in inds(t)])
    total_params += params
    println("Tensor $i has $params parameters")
end
println("Total parameters in tensor train: $total_params")

In [ ]:
# Demonstrating tensor train operations

println("Tensor Train Operations:")
println("="^40)

# Contract the entire tensor train to get the full tensor
println("1. Contracting the entire tensor train...")
full_tensor = mps_tensors[1]
for i in 2:length(mps_tensors)
    full_tensor = full_tensor * mps_tensors[i]
end

println("Full tensor indices: $(inds(full_tensor))")
println("Full tensor would have $(2^N) = $(2^N) elements if stored densely")
println("Our tensor train uses only $total_params parameters!")

# Calculate compression ratio
dense_params = 2^N
compression_ratio = dense_params / total_params
println("Compression ratio: $(round(compression_ratio, digits=2))x")

println("\n2. Accessing specific tensor elements...")
# Show how to access elements (this is just an example of indexing)
val = full_tensor[sites[1]=>1, sites[2]=>1, sites[3]=>1, sites[4]=>1, sites[5]=>1, sites[6]=>1]
println("Element |000000⟩: $val")

val2 = full_tensor[sites[1]=>2, sites[2]=>2, sites[3]=>2, sites[4]=>2, sites[5]=>2, sites[6]=>2]
println("Element |111111⟩: $val2")

# Calculate the norm of the tensor train
tensor_norm = norm(full_tensor)
println("\n3. Norm of the tensor train: $tensor_norm")

## Tensor Trains (Matrix Product States)

### What is a Tensor Train?

A **Tensor Train (TT)** or **Matrix Product State (MPS)** is a factorization of a high-dimensional tensor into a chain of lower-dimensional tensors connected by shared bond indices.

For a tensor $T$ with $N$ indices, the tensor train decomposition is:
$$T_{i_1,i_2,\ldots,i_N} = \sum_{\alpha_1,\alpha_2,\ldots,\alpha_{N-1}} A^{(1)}_{i_1,\alpha_1} A^{(2)}_{\alpha_1,i_2,\alpha_2} \cdots A^{(N)}_{\alpha_{N-1},i_N}$$

Where:
- $A^{(k)}$ are the tensor train cores
- $\alpha_k$ are bond indices with dimension $\chi$ (bond dimension)
- $i_k$ are physical indices with dimension $d$

### Key Advantages:

1. **Memory Efficiency**: Instead of storing $d^N$ parameters, we store $\mathcal{O}(Nd\chi^2)$ parameters
2. **Computational Efficiency**: Many operations can be performed directly on the TT format
3. **Controlled Approximation**: Bond dimension $\chi$ controls accuracy vs efficiency trade-off
4. **Structure Preservation**: Natural for quantum many-body systems and PDEs

### In Our Example:
- **6 sites** with **dimension 2** each → Full tensor: $2^6 = 64$ elements  
- **Bond dimension 4** → Tensor train: 144 parameters
- The compression ratio depends on the bond dimension and can be much better for larger systems!

In [ ]:
using Pkg; Pkg.activate(".")
using Dleto
using Plots

In [ ]:
T, Xs = randomize_tensor(full_tensor)
@time S, Xs = stratify(T)